## LangGraph

**LangGraph-> is a graph orchestration framework on the top the langchain, where your app is modeled as nodes(functions) and edges(transitions),with shared state flowing through.In real agent  for control flow we need branching, looping and retrying until the result gives success**

#### Part 1: **State-the heart of the langgraph**

In [5]:
from typing import TypedDict, Annotated
from operator import add

class AgentState(TypedDict):
    messages: Annotated[list,add] # add= reducer, it appends the messages not overwrite it
    question: str
    answer: str

#### Part-2: **Node**

In [6]:
def get_question(state:AgentState)-> AgentState: 
    return {"question":"What is langgraph"}

def answer_question(state:AgentState)-> AgentState:
    ans = f"Answer to :{state['question']}"
    return {"answer":ans}

#node function always return dict with only keys it wants to update - not the full state 

#### Part 3: **Building the state Graph**

In [7]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("get_q",get_question)
builder.add_node("get_ans",answer_question)

builder.add_edge(START,"get_q")
builder.add_edge("get_q","get_ans") # edges are the transistion state for langgraph
builder.add_edge("get_ans",END)

graph = builder.compile()

result = graph.invoke({"messages":[], "question":"","answer":""})
print(result)

{'messages': [], 'question': 'What is langgraph', 'answer': 'Answer to :What is langgraph'}


#### Part 4: **Conditional Edges**

In [8]:
# Conditional edges are branching the graph besed on the required contions in the graph
# first we have to create the function node with the conditions
def route_decision(state:AgentState):
    if "quantum" in state["question"]:
        return "quantum_node"
    else:
        return "normal_node"

builder.add_conditional_edges(
    "get_q", # source node
    route_decision, # function returning next node name 
    {
        "quantum_node":"quantum_node",
        "normal_node":"normal_node"
    }
    )

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


#### Part 5: **Loops**

In [9]:
def check_quality(state:AgentState):
    if len(state["answer"]) < 20:
        return "retry"
    return "done"


builder.add_conditional_edges(
    "get_q",
    check_quality,
    {
        "retry" : "get_ans",  # loop back it self
        "done" : END
    }
    )

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


#### Part 7: **Checkpointing/Persistence (memory)**

In [10]:
from langgraph.checkpoint.memory import MemorySaver

def get_question(state:AgentState)-> AgentState: 
    return {"question":"What is langgraph"}

def answer_question(state:AgentState)-> AgentState:
    ans = f"Answer to :{state['question']}"
    return {"answer":ans}

builder = StateGraph(AgentState)

builder.add_node("get_q",get_question)
builder.add_node("get_ans",answer_question)

builder.add_edge(START,"get_q")
builder.add_edge("get_q","get_ans") # edges are the transistion state for langgraph
builder.add_edge("get_ans",END)

checkpointer = MemorySaver()  ## check pointer is hereeeeee

graph = builder.compile(checkpointer=checkpointer)

config = {"configurable":{"thread_id":"user-123"}}

# first call
graph.invoke({"messages":[{"role":"user","content":"Hi"}]},config=config)

# second call also can use the same config with thread-id for passing the old info
graph.invoke({"messages":[{"role":"user","content":"What did i say?"}]},config=config)

{'messages': [{'role': 'user', 'content': 'Hi'},
  {'role': 'user', 'content': 'What did i say?'}],
 'question': 'What is langgraph',
 'answer': 'Answer to :What is langgraph'}

##### for production use **SqliteSaver** or **PostgreSaver**

In [1]:
# from langgraph.checkpoint.sqlite import SqliteSaver

#checkpointer = SqliteSaver.from_conn_string("Checkpoint.db")